In [ ]:
!tar -xf /kaggle/input/datasets/dschettler8845/brats-2021-task1/BraTS2021_Training_Data.tar -C /kaggle/working/


In [ ]:
import os
import pandas as pd

# Load your manifest file
df = pd.read_csv("/kaggle/input/datasets/tejaswinirj/data-upd/manifest.csv")

# Define the target directory on Kaggle
new_base_dir = "/kaggle/input/datasets/tejaswinirj/data-upd"


# Function to handle Windows paths and extract the file name
def update_path(old_path):
    if pd.isna(old_path):
        return old_path

    # Replace backslashes with forward slashes for cross-platform safety
    clean_path = str(old_path).replace("\\", "/")

    # Extract the filename (e.g., 'BraTS2021_00000.npz')
    filename = os.path.basename(clean_path)

    # Combine into the new Kaggle directory format
    return f"{new_base_dir}/{filename}"


# Update the 'data_path' column
df["data_path"] = df["data_path"].apply(update_path)

# Save the updated manifest back to a CSV
df.to_csv("manifest_updated.csv", index=False)

print("Successfully updated 1251 file paths!")

In [ ]:
import os
import shutil

base = "/kaggle/working"
target = os.path.join(base, "BraTS2021_Training_Data")

os.makedirs(target, exist_ok=True)

for d in os.listdir(base):
    if d.startswith("BraTS2021_"):
        shutil.move(os.path.join(base, d), target)

print("Done")

In [1]:
import os
import warnings
from pathlib import Path
from typing import Dict, Tuple, List
 
import json
import numpy as np
import pandas as pd
import glob
import re
import gzip
import shutil
import nibabel as nib
import tensorflow as tf
from tensorflow import keras
import gc
from tensorflow.keras import layers, Model
from tensorflow.keras import backend as K
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.layers import (
    Conv3D, MaxPooling3D, Conv3DTranspose, Input,
    BatchNormalization, Activation, SpatialDropout3D, concatenate
)
from tensorflow.keras import regularizers, initializers
from sklearn.model_selection import train_test_split

In [2]:
warnings.filterwarnings('ignore')
USE_FLOAT16 = True
MODALITIES = ["t1", "t1ce", "t2", "flair"]
 
 
# ---------------------------------------------------------------------
# Manifest path fix-up (unchanged from your version)
# ---------------------------------------------------------------------
def fixup_manifest_paths(manifest_in="/kaggle/input/private-dataset/manifest.csv",
                          new_base_dir="/kaggle/input/private-dataset",
                          manifest_out="manifest_updated.csv"):
    df_fix = pd.read_csv(manifest_in)
 
    def update_path(old_path):
        if pd.isna(old_path):
            return old_path
        clean_path = str(old_path).replace("\\", "/")
        filename = os.path.basename(clean_path)
        return f"{new_base_dir}/{filename}"
 
    df_fix["data_path"] = df_fix["data_path"].apply(update_path)
    df_fix.to_csv(manifest_out, index=False)
    print(f"Successfully updated {len(df_fix)} file paths!")
 
 
def move_brats_folders(base="/kaggle/working", target_name="BraTS2021_Training_Data"):
    target = os.path.join(base, target_name)
    os.makedirs(target, exist_ok=True)
    for d in os.listdir(base):
        if d.startswith("BraTS2021_"):
            shutil.move(os.path.join(base, d), target)
    print("Done")
 
 
# ---------------------------------------------------------------------
# Loading
# ---------------------------------------------------------------------
def load_nifti_canonical(path: Path) -> Tuple[np.ndarray, np.ndarray, dict]:
    nii = nib.as_closest_canonical(nib.load(str(path)))
    data = nii.get_fdata(dtype=np.float32)
    affine = nii.affine
    header = {k: nii.header[k].tolist() if k in nii.header else None
              for k in ["dim", "pixdim"]}
    return data, affine, header
 
 
def normalize_nonzero(volume: np.ndarray) -> np.ndarray:
    vol = volume.astype(np.float32)
    nz = vol[vol != 0]
    if nz.size > 0:
        mean, std = float(nz.mean()), float(nz.std())
    else:
        mean, std = float(vol.mean()), float(vol.std())
    return (vol - mean) / (std + 1e-8)
 
 
def preprocess_mask_multiclass(mask: np.ndarray) -> np.ndarray:
    m = np.rint(mask).astype(np.uint8)
    m[m == 4] = 3
    return m
 
 
# ---------------------------------------------------------------------
# FIX #1 + FIX #2: crop using the IMAGE, never the ground-truth mask.
# ---------------------------------------------------------------------
def crop_to_brain_bbox(
    image_4ch: np.ndarray,
    mask: np.ndarray = None,
    margin: int = 5,
    intensity_percentile: float = 1.0,
) -> Tuple[np.ndarray, np.ndarray]:
    signal = np.abs(image_4ch).sum(axis=-1)
    thresh = np.percentile(signal[signal > 0], intensity_percentile) if np.any(signal > 0) else 0
    fg = signal > thresh
 
    coords = np.array(np.nonzero(fg))
    if coords.size == 0:
        return image_4ch, mask
 
    zmin, ymin, xmin = coords.min(axis=1)
    zmax, ymax, xmax = coords.max(axis=1)
 
    zmin = max(zmin - margin, 0)
    ymin = max(ymin - margin, 0)
    xmin = max(xmin - margin, 0)
    zmax = min(zmax + margin, image_4ch.shape[0] - 1)
    ymax = min(ymax + margin, image_4ch.shape[1] - 1)
    xmax = min(xmax + margin, image_4ch.shape[2] - 1)
 
    img_c = image_4ch[zmin:zmax + 1, ymin:ymax + 1, xmin:xmax + 1]
    msk_c = mask[zmin:zmax + 1, ymin:ymax + 1, xmin:xmax + 1] if mask is not None else None
    return img_c, msk_c
 
 
def process_one_patient(patient_folder: Path, patient_id: str) -> Tuple[np.ndarray, np.ndarray, Dict]:
    mask_path = patient_folder / f"{patient_id}_seg.nii.gz"
    if not mask_path.exists():
        raise FileNotFoundError(f"Mask file not found: {mask_path}")
    mask_vol, aff_m, hdr_m = load_nifti_canonical(mask_path)
    mask = preprocess_mask_multiclass(mask_vol)
 
    vols = []
    metas = {"patient_id": patient_id, "modalities": MODALITIES}
    for mod in MODALITIES:
        p = patient_folder / f"{patient_id}_{mod}.nii.gz"
        if not p.exists():
            raise FileNotFoundError(f"File not found: {p}")
        vol, aff, hdr = load_nifti_canonical(p)
        vols.append(vol)
        metas[f"affine_{mod}"] = aff.tolist()
        metas[f"header_{mod}"] = hdr
 
    shapes = [v.shape for v in vols]
    if len(set(shapes)) != 1:
        raise ValueError(f"[AlignmentError] Modality shapes differ for patient {patient_id}: {shapes}")
 
    image_4ch = np.stack(vols, axis=-1)
    image_4ch, mask = crop_to_brain_bbox(image_4ch, mask)
 
    for i in range(image_4ch.shape[-1]):
        image_4ch[..., i] = normalize_nonzero(image_4ch[..., i])
 
    if USE_FLOAT16:
        image_4ch = image_4ch.astype(np.float16)
 
    metas["affine_mask"] = aff_m.tolist()
    metas["header_mask"] = hdr_m
    metas["shape"] = list(mask.shape)
    return image_4ch, mask, metas
 
 
def save_patient_npz(save_dir: Path, patient_id: str, image_4ch: np.ndarray, mask: np.ndarray) -> Path:
    out_path = save_dir / f"{patient_id}.npz"
    np.savez_compressed(out_path, image=image_4ch, mask=mask)
    return out_path

In [3]:
# ---------------------------------------------------------------------
# Patch extraction / augmentation
# ---------------------------------------------------------------------
def pad_to_min_shape(arr, min_shape, pad_value=0):
    pad_width = []
    for i, s in enumerate(arr.shape[:3]):
        needed = max(0, min_shape[i] - s)
        before = needed // 2
        after = needed - before
        pad_width.append((before, after))
    if arr.ndim == 4:
        pad_width.append((0, 0))
    return np.pad(arr, pad_width, mode='constant', constant_values=pad_value)
 
 
def extract_patch(image, mask, patch_size, center=None):
    h, w, d = image.shape[:3]
    ph, pw, pd = patch_size
 
    if h < ph or w < pw or d < pd:
        image = pad_to_min_shape(image, patch_size, pad_value=0)
        mask = pad_to_min_shape(mask, patch_size, pad_value=0)
        h, w, d = image.shape[:3]
 
    if center is None:
        y = (h - ph) // 2
        x = (w - pw) // 2
        z = (d - pd) // 2
    else:
        cy, cx, cz = center
        y = int(np.clip(cy - ph // 2, 0, h - ph))
        x = int(np.clip(cx - pw // 2, 0, w - pw))
        z = int(np.clip(cz - pd // 2, 0, d - pd))
 
    img_patch = image[y:y + ph, x:x + pw, z:z + pd]
    msk_patch = mask[y:y + ph, x:x + pw, z:z + pd]
    return img_patch, msk_patch
 
 
def random_flip_3d(img, msk):
    for axis in [0, 1, 2]:
        if np.random.rand() < 0.5:
            img = np.flip(img, axis=axis)
            msk = np.flip(msk, axis=axis)
    return img, msk
 
 
def intensity_jitter(img, shift_limit=0.1, scale_limit=0.1):
    out = img.astype(np.float32)
    for c in range(out.shape[-1]):
        scale = 1.0 + np.random.uniform(-scale_limit, scale_limit)
        shift = np.random.uniform(-shift_limit, shift_limit)
        out[..., c] = out[..., c] * scale + shift
    return out
 
 
def add_noise(img, std=0.02):
    noise = np.random.normal(0, std, size=img.shape).astype(np.float32)
    return img + noise
 
 
# ---------------------------------------------------------------------
# FIX #3 + FIX #5: generator
# ---------------------------------------------------------------------
class BraTSDataGenerator(tf.keras.utils.Sequence):
    def __init__(self, df, patch_size, batch_size=1, is_train=False,
                 augment=False, shuffle=True, min_tumor_frac=0.01):
        self.df = df.reset_index(drop=True)
        self.patch_size = patch_size
        self.batch_size = batch_size
        self.is_train = is_train
        self.augment = augment
        self.shuffle = shuffle
        self.min_tumor_frac = min_tumor_frac
        self.on_epoch_end()
 
    def __len__(self):
        return int(np.floor(len(self.df) / self.batch_size))
 
    def on_epoch_end(self):
        self.indexes = np.arange(len(self.df))
        if self.shuffle:
            np.random.shuffle(self.indexes)
 
    def __getitem__(self, index):
        idxs = self.indexes[index * self.batch_size:(index + 1) * self.batch_size]
        X = np.zeros((self.batch_size, *self.patch_size, 4), dtype=np.float32)
        y = np.zeros((self.batch_size, *self.patch_size, 1), dtype=np.uint8)
 
        for i, ridx in enumerate(idxs):
            row = self.df.iloc[ridx]
            with np.load(row['data_path']) as npz:
                image = npz['image'].astype(np.float32)
                mask = npz['mask'].astype(np.uint8)
 
            patch_img, patch_msk = None, None
 
            if self.is_train:
                tries = 0
                while tries < 10:
                    tumor_coords = np.argwhere(mask > 0)
                    if tumor_coords.size > 0:
                        cy, cx, cz = tumor_coords[np.random.randint(0, len(tumor_coords))]
                        center = (cy, cx, cz)
                    else:
                        center = None
                    patch_img, patch_msk = extract_patch(image, mask, self.patch_size, center=center)
                    frac = (patch_msk > 0).mean()
                    if frac >= self.min_tumor_frac or tumor_coords.size == 0:
                        break
                    tries += 1
            else:
                patch_img, patch_msk = extract_patch(image, mask, self.patch_size, center=None)
 
            if self.augment:
                patch_img, patch_msk = random_flip_3d(patch_img, patch_msk)
                if np.random.rand() < 0.5:
                    patch_img = intensity_jitter(patch_img)
                if np.random.rand() < 0.5:
                    patch_img = add_noise(patch_img)
 
            X[i] = patch_img
            y[i] = np.expand_dims(patch_msk, axis=-1)
 
        return X, y

In [4]:
# ---------------------------------------------------------------------
# 3D U-Net (baselines removed — this is the only architecture now)
# ---------------------------------------------------------------------
def conv_block(inputs, num_filters, kernel_size=3, initializer='he_normal', regularizer=None):
    x = layers.Conv3D(num_filters, kernel_size, padding='same',
                       kernel_initializer=initializer, kernel_regularizer=regularizer)(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Conv3D(num_filters, kernel_size, padding='same',
                       kernel_initializer=initializer, kernel_regularizer=regularizer)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    return x
 
 
def decoder_block(inputs, skip_features, num_filters, kernel_size=3, strides=2,
                   initializer='he_normal', regularizer=None):
    x = layers.Conv3DTranspose(num_filters, kernel_size, strides=(strides, strides, strides),
                                padding='same', kernel_initializer=initializer,
                                kernel_regularizer=regularizer)(inputs)
    x = layers.concatenate([x, skip_features])
    x = conv_block(x, num_filters, kernel_size, initializer, regularizer)
    return x
 
 
def create_advanced_3d_unet(input_shape, num_classes=4):
    reg = regularizers.l2(1e-5)
    inputs = layers.Input(input_shape)
 
    s1 = conv_block(inputs, 32, regularizer=reg)
    p1 = SpatialDropout3D(0.05)(layers.MaxPooling3D((2, 2, 2))(s1))
 
    s2 = conv_block(p1, 64, regularizer=reg)
    p2 = SpatialDropout3D(0.1)(layers.MaxPooling3D((2, 2, 2))(s2))
 
    s3 = conv_block(p2, 128, regularizer=reg)
    p3 = SpatialDropout3D(0.15)(layers.MaxPooling3D((2, 2, 2))(s3))
 
    s4 = conv_block(p3, 256, regularizer=reg)
    p4 = SpatialDropout3D(0.2)(layers.MaxPooling3D((2, 2, 2))(s4))
 
    b1 = conv_block(p4, 512, regularizer=reg)
 
    d1 = decoder_block(b1, s4, 256, regularizer=reg)
    d2 = decoder_block(d1, s3, 128, regularizer=reg)
    d3 = decoder_block(d2, s2, 64, regularizer=reg)
    d4 = decoder_block(d3, s1, 32, regularizer=reg)
 
    outputs = layers.Conv3D(num_classes, 1, activation='softmax')(d4)
    model = Model(inputs=[inputs], outputs=[outputs], name='Advanced_3D_U-Net')
    return model

In [5]:
# ---------------------------------------------------------------------
# Losses / metrics
# ---------------------------------------------------------------------
SMOOTH = 1e-6
 
 
def dice_coef(y_true, y_pred):
    y_true_f = K.flatten(y_true)
    y_pred_f = K.flatten(y_pred)
    inter = K.sum(y_true_f * y_pred_f)
    return (2. * inter + SMOOTH) / (K.sum(y_true_f) + K.sum(y_pred_f) + SMOOTH)
 
 
def combined_dice_ce_loss(y_true, y_pred):
    y_true = tf.squeeze(y_true, axis=-1)
    num_classes = K.int_shape(y_pred)[-1]
    y_true_oh = tf.one_hot(tf.cast(y_true, tf.int32), depth=num_classes)
    dice_sum = 0.0
    for c in range(1, num_classes):
        dice_sum += dice_coef(y_true_oh[..., c], y_pred[..., c])
    dice_mean = dice_sum / tf.cast(num_classes - 1, tf.float32)
    ce = K.categorical_crossentropy(y_true_oh, y_pred)
    ce_mean = K.mean(ce)
    return 0.5 * (1.0 - dice_mean) + 0.5 * ce_mean
 
 
def iou_coef(y_true, y_pred, smooth=1e-6):
    y_true = tf.squeeze(y_true, axis=-1)
    num_classes = K.int_shape(y_pred)[-1]
    y_true_oh = tf.one_hot(tf.cast(y_true, tf.int32), depth=num_classes)
    y_pred_arg = K.argmax(y_pred, axis=-1)
    y_pred_oh = tf.one_hot(y_pred_arg, depth=num_classes)
    intersection = K.sum(y_true_oh * y_pred_oh, axis=[0, 1, 2, 3])
    union = K.sum(y_true_oh + y_pred_oh, axis=[0, 1, 2, 3]) - intersection
    return K.mean((intersection + smooth) / (union + smooth))
 
 
def dice_whole_tumor(y_true, y_pred):
    y_true = tf.squeeze(y_true, axis=-1)
    num_classes = K.int_shape(y_pred)[-1]
    y_true_oh = tf.one_hot(tf.cast(y_true, tf.int32), depth=num_classes)
    y_pred_arg = K.argmax(y_pred, axis=-1)
    y_true_wt = K.cast(K.any(y_true_oh[..., 1:], axis=-1), 'float32')
    y_pred_wt = K.cast(K.any(tf.stack([tf.equal(y_pred_arg, 1), tf.equal(y_pred_arg, 2),
                                        tf.equal(y_pred_arg, 3)], axis=-1), axis=-1), 'float32')
    return dice_coef(y_true_wt, y_pred_wt)
 
 
def dice_tumor_core(y_true, y_pred):
    y_true = tf.squeeze(y_true, axis=-1)
    num_classes = K.int_shape(y_pred)[-1]
    y_true_oh = tf.one_hot(tf.cast(y_true, tf.int32), depth=num_classes)
    y_pred_arg = K.argmax(y_pred, axis=-1)
    y_true_tc = K.cast(K.any(tf.stack([y_true_oh[..., 1], y_true_oh[..., 3]], axis=-1), axis=-1), 'float32')
    y_pred_tc = K.cast(K.any(tf.stack([tf.equal(y_pred_arg, 1), tf.equal(y_pred_arg, 3)], axis=-1), axis=-1), 'float32')
    return dice_coef(y_true_tc, y_pred_tc)
 
 
def dice_enhancing_tumor(y_true, y_pred):
    y_true = tf.squeeze(y_true, axis=-1)
    num_classes = K.int_shape(y_pred)[-1]
    y_true_oh = tf.one_hot(tf.cast(y_true, tf.int32), depth=num_classes)
    y_pred_arg = K.argmax(y_pred, axis=-1)
    y_true_et = y_true_oh[..., 3]
    y_pred_et = K.cast(tf.equal(y_pred_arg, 3), 'float32')
    return dice_coef(y_true_et, y_pred_et)

In [19]:
# =======================================================================
# Full-volume sliding-window inference — this is what produces the
# reviewer-requested WT/TC/ET Dice/IoU/Precision/Recall numbers, not the
# training-time patch metrics above.
# =======================================================================
def sliding_window_predict(model, volume_4ch, patch_size, stride, num_classes, verbose=False):
    orig_shape = volume_4ch.shape[:3]
    ph, pw, pd = patch_size
    sh, sw, sd = stride
 
    padded = pad_to_min_shape(volume_4ch, patch_size, pad_value=0)
    H, W, D = padded.shape[:3]
 
    def stops(dim_size, patch, step):
        if dim_size <= patch:
            return [0]
        s = list(range(0, dim_size - patch + 1, step))
        if s[-1] != dim_size - patch:
            s.append(dim_size - patch)
        return s
 
    y_stops = stops(H, ph, sh)
    x_stops = stops(W, pw, sw)
    z_stops = stops(D, pd, sd)
 
    prob_sum = np.zeros((H, W, D, num_classes), dtype=np.float32)
    count = np.zeros((H, W, D, 1), dtype=np.float32)
 
    total = len(y_stops) * len(x_stops) * len(z_stops)
    done = 0
    for y in y_stops:
        for x in x_stops:
            for z in z_stops:
                patch = padded[y:y + ph, x:x + pw, z:z + pd]
                pred = model.predict(patch[np.newaxis, ...].astype(np.float32), verbose=0)[0]
                prob_sum[y:y + ph, x:x + pw, z:z + pd] += pred
                count[y:y + ph, x:x + pw, z:z + pd] += 1.0
                done += 1
                if verbose and done % 5 == 0:
                    print(f"  sliding window patch {done}/{total}")
 
    count[count == 0] = 1.0
    avg_prob = prob_sum / count
    label_full = np.argmax(avg_prob, axis=-1).astype(np.uint8)
    label_full = label_full[:orig_shape[0], :orig_shape[1], :orig_shape[2]]
    return label_full
 
 
def region_mask_np(label_volume: np.ndarray, region: str) -> np.ndarray:
    if region == "WT":
        return np.isin(label_volume, [1, 2, 3])
    if region == "TC":
        return np.isin(label_volume, [1, 3])
    if region == "ET":
        return label_volume == 3
    raise ValueError(f"Unknown region: {region}")
 
 
def dice_iou_precision_recall_np(pred_mask: np.ndarray, gt_mask: np.ndarray, smooth: float = 1e-6):
    pred_mask = pred_mask.astype(bool)
    gt_mask = gt_mask.astype(bool)
    tp = np.logical_and(pred_mask, gt_mask).sum()
    fp = np.logical_and(pred_mask, ~gt_mask).sum()
    fn = np.logical_and(~pred_mask, gt_mask).sum()
 
    dice = (2.0 * tp + smooth) / (2.0 * tp + fp + fn + smooth)
    iou = (tp + smooth) / (tp + fp + fn + smooth)
    precision = (tp + smooth) / (tp + fp + smooth)
    recall = (tp + smooth) / (tp + fn + smooth)
    return dice, iou, precision, recall
 
 
def evaluate_patient_full_volume(model, volume_4ch, gt_label, patch_size, stride, num_classes):
    pred_label = sliding_window_predict(model, volume_4ch, patch_size, stride, num_classes)
    result = {}
    for region in ("WT", "TC", "ET"):
        pred_mask = region_mask_np(pred_label, region)
        gt_mask = region_mask_np(gt_label, region)
        dice, iou, prec, rec = dice_iou_precision_recall_np(pred_mask, gt_mask)
        result[f"{region}_Dice"] = dice
        result[f"{region}_IoU"] = iou
        result[f"{region}_Precision"] = prec
        result[f"{region}_Recall"] = rec
    return result
 
 
def evaluate_full_dataset(model, df: pd.DataFrame, patch_size, stride, num_classes) -> pd.DataFrame:
    rows = []
    for _, row in df.iterrows():
        with np.load(row['data_path']) as npz:
            image = npz['image'].astype(np.float32)
            mask = npz['mask'].astype(np.uint8)
        metrics = evaluate_patient_full_volume(model, image, mask, patch_size, stride, num_classes)
        metrics['patient_id'] = row['patient_id']
        rows.append(metrics)
    results_df = pd.DataFrame(rows)
    cols = ['patient_id'] + [c for c in results_df.columns if c != 'patient_id']
    return results_df[cols]
 
 
def summarize_results(results_df: pd.DataFrame) -> pd.DataFrame:
    metric_cols = [c for c in results_df.columns if c != 'patient_id']
    summary = pd.DataFrame({
        'mean': results_df[metric_cols].mean(),
        'std': results_df[metric_cols].std(),
        'median': results_df[metric_cols].median(),
    })
    return summary
 
 
def format_region_metric_table(summary_df: pd.DataFrame, use_std: bool = True) -> pd.DataFrame:
    """
    Reshapes summary_df (indexed by 'WT_Dice', 'WT_IoU', ... with
    mean/std/median columns) into the reviewer-requested table format:
 
        Region                  Dice    IoU    Precision   Recall
        Whole Tumor (WT)        ...     ...    ...         ...
        Tumor Core (TC)         ...     ...    ...         ...
        Enhancing Tumor (ET)    ...     ...    ...         ...
 
    Each cell is "mean ± std" (use_std=True) or just "mean" (use_std=False).
    """
    region_display_names = {
        "WT": "Whole Tumor (WT)",
        "TC": "Tumor Core (TC)",
        "ET": "Enhancing Tumor (ET)",
    }
    metric_order = ["Dice", "IoU", "Precision", "Recall"]
 
    rows = []
    for region_code, display_name in region_display_names.items():
        row = {"Region": display_name}
        for metric in metric_order:
            key = f"{region_code}_{metric}"
            mean_val = summary_df.loc[key, "mean"]
            if use_std:
                std_val = summary_df.loc[key, "std"]
                row[metric] = f"{mean_val:.4f} ± {std_val:.4f}"
            else:
                row[metric] = f"{mean_val:.4f}"
        rows.append(row)
 
    table = pd.DataFrame(rows).set_index("Region")
    return table
 
 
def save_results_excel(path: str, patient_df: pd.DataFrame, summary_df: pd.DataFrame):
    """Reviewer-requested output as a table, saved to Excel (not CSV)."""
    region_table = format_region_metric_table(summary_df, use_std=True)
    with pd.ExcelWriter(path, engine='openpyxl') as writer:
        patient_df.to_excel(writer, sheet_name='per_patient', index=False)
        summary_df.to_excel(writer, sheet_name='summary')
        region_table.to_excel(writer, sheet_name='region_table')
 

In [20]:
# =======================================================================
# LIVE LOGGING: update the results sheet as training progresses instead
# of only writing it once at the very end.
# =======================================================================
class EpochMetricsExcelLogger(tf.keras.callbacks.Callback):
    """
    Cheap, runs every epoch: appends the epoch's training/validation
    metrics (loss, dice, iou, etc. already computed by Keras — no extra
    forward passes) to an Excel sheet. This is what keeps a live,
    continuously-updated training log even if the session gets
    interrupted mid-run.
    """
    def __init__(self, excel_path, start_epoch=0):
        super().__init__()
        self.excel_path = excel_path
        if os.path.exists(excel_path):
            self.log_df = pd.read_excel(excel_path)
        else:
            self.log_df = pd.DataFrame()
 
    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        row = {'epoch': epoch + 1}
        row.update({k: float(v) for k, v in logs.items()})
        self.log_df = pd.concat([self.log_df, pd.DataFrame([row])], ignore_index=True)
        self.log_df.to_excel(self.excel_path, index=False)
 
 
class PeriodicFullVolumeEvalLogger(tf.keras.callbacks.Callback):
    """
    Runs full sliding-window WT/TC/ET evaluation on the validation set
    every `eval_every` epochs (this is the expensive part — many forward
    passes per patient — so it is NOT run every single epoch) and
    overwrites the results Excel file each time. This means the
    reviewer-requested results table is kept up to date throughout
    training, not just written once after the final epoch.
    """
    def __init__(self, val_df, patch_size, stride, num_classes, excel_path, eval_every=5):
        super().__init__()
        self.val_df = val_df
        self.patch_size = patch_size
        self.stride = stride
        self.num_classes = num_classes
        self.excel_path = excel_path
        self.eval_every = eval_every
 
    def on_epoch_end(self, epoch, logs=None):
        current_epoch = epoch + 1
        if current_epoch % self.eval_every != 0:
            return
        print(f"\n[epoch {current_epoch}] Running periodic full-volume validation evaluation...")
        val_results = evaluate_full_dataset(self.model, self.val_df, self.patch_size,
                                             self.stride, self.num_classes)
        val_summary = summarize_results(val_results)
        save_results_excel(self.excel_path, val_results, val_summary)
        print(f"[epoch {current_epoch}] Updated {self.excel_path}")
        print(val_summary)
        del val_results, val_summary
        gc.collect()

In [21]:
# ---------------------------------------------------------------------
# Preprocessing driver
# ---------------------------------------------------------------------
DATA_DIR = Path(r"/kaggle/working/BraTS2021_Training_Data")
SAVE_DIR = Path(r"/kaggle/input/datasets/tejaswinirj/data-upd")
SAVE_DIR.mkdir(parents=True, exist_ok=True)
 
 
def run_preprocessing(data_dir: Path, save_dir: Path) -> pd.DataFrame:
    patient_folders = [p for p in data_dir.iterdir() if p.is_dir()]
    records = []
    for p_folder in patient_folders:
        patient_id = p_folder.name
        try:
            img, msk, meta = process_one_patient(p_folder, patient_id)
            data_path = save_patient_npz(save_dir, patient_id, img, msk)
            records.append({'patient_id': patient_id, 'data_path': str(data_path)})
        except Exception as e:
            print(f"Error processing patient {patient_id}: {e}")
    return pd.DataFrame(records)

In [22]:
preprocessed_manifest = Path("/kaggle/working/manifest_updated.csv")
if not preprocessed_manifest.exists():
    print("Running preprocessing...")
    df = run_preprocessing(DATA_DIR, SAVE_DIR)
    df.to_csv(preprocessed_manifest, index=False)
else:
    print("Loading preprocessed manifest...")
    df = pd.read_csv(preprocessed_manifest)

print(f"Total patients: {len(df)}")

Loading preprocessed manifest...
Total patients: 1250


In [23]:
CONFIG = {
    "dataset": "BraTS",
    "modalities": MODALITIES,
    "seed": 42,
    "train_ratio": 0.70,
    "val_ratio": 0.15,
    "test_ratio": 0.15,
    "patch_size": (112, 112, 112),
    "inference_stride": (56, 56, 56),   # 50% overlap sliding window
    "batch_size": 1,
    "num_classes": 4,
    "epochs": 40,
    "optimizer": "Adam",
    "learning_rate": 1e-4,
    "loss": "0.5*(1-Dice) + 0.5*CategoricalCrossEntropy",
    "augmentation": {"flip": True, "intensity_jitter": True, "gaussian_noise": True},
    "eval_every_n_epochs": 5,  # how often the live full-volume val table refreshes
}
print("Reproducibility config:", CONFIG)

PATCH_SIZE = CONFIG["patch_size"]
STRIDE = CONFIG["inference_stride"]
BATCH_SIZE = CONFIG["batch_size"]
NUM_CLASSES = CONFIG["num_classes"]
EPOCHS = CONFIG["epochs"]
SEED = CONFIG["seed"]
EVAL_EVERY = CONFIG["eval_every_n_epochs"]

train_df, temp_df = train_test_split(
    df, test_size=(1 - CONFIG["train_ratio"]), random_state=SEED, shuffle=True
)
rel_test = CONFIG["test_ratio"] / (CONFIG["val_ratio"] + CONFIG["test_ratio"])
val_df, test_df = train_test_split(temp_df, test_size=rel_test, random_state=SEED, shuffle=True)
print(f"Train size: {len(train_df)}, Validation size: {len(val_df)}, Test size: {len(test_df)}")

train_gen = BraTSDataGenerator(train_df, patch_size=PATCH_SIZE, batch_size=BATCH_SIZE,
                                is_train=True, augment=True, shuffle=True)
val_gen = BraTSDataGenerator(val_df, patch_size=PATCH_SIZE, batch_size=BATCH_SIZE,
                              is_train=False, augment=False, shuffle=False)
test_gen = BraTSDataGenerator(test_df, patch_size=PATCH_SIZE, batch_size=BATCH_SIZE,
                               is_train=False, augment=False, shuffle=False)

Reproducibility config: {'dataset': 'BraTS', 'modalities': ['t1', 't1ce', 't2', 'flair'], 'seed': 42, 'train_ratio': 0.7, 'val_ratio': 0.15, 'test_ratio': 0.15, 'patch_size': (112, 112, 112), 'inference_stride': (56, 56, 56), 'batch_size': 1, 'num_classes': 4, 'epochs': 40, 'optimizer': 'Adam', 'learning_rate': 0.0001, 'loss': '0.5*(1-Dice) + 0.5*CategoricalCrossEntropy', 'augmentation': {'flip': True, 'intensity_jitter': True, 'gaussian_noise': True}, 'eval_every_n_epochs': 5}
Train size: 874, Validation size: 188, Test size: 188


In [24]:
loss_fn = combined_dice_ce_loss
train_metrics = [dice_whole_tumor, dice_tumor_core, dice_enhancing_tumor, iou_coef]
custom_objects = {
    'combined_dice_ce_loss': combined_dice_ce_loss,
    'dice_whole_tumor': dice_whole_tumor,
    'dice_tumor_core': dice_tumor_core,
    'dice_enhancing_tumor': dice_enhancing_tumor,
    'iou_coef': iou_coef,
}

In [12]:
model_key = "unet"
print(f"\n{'=' * 70}\nTraining architecture: {model_key}\n{'=' * 70}")

BEST_MODEL_PATH = f"/kaggle/working/best_{model_key}_model.keras"
EPOCH_CKPT_PATTERN = f"/kaggle/working/{model_key}_full_epoch_{{epoch:03d}}.keras.gz"
EPOCH_CKPT_GLOB = f"/kaggle/working/{model_key}_full_epoch_*.keras.gz"
EPOCH_CKPT_REGEX = re.compile(rf"{model_key}_full_epoch_(\d+)\.keras\.gz$")

# --- Resume code retained exactly as before ------------------------
existing_ckpts = glob.glob(EPOCH_CKPT_GLOB)
initial_epoch = 0
if existing_ckpts:
    def _epoch_num(path):
        m = EPOCH_CKPT_REGEX.search(path)
        return int(m.group(1)) if m else -1

    latest_ckpt = max(existing_ckpts, key=_epoch_num)
    initial_epoch = _epoch_num(latest_ckpt)
    print(f"Found existing checkpoint '{latest_ckpt}' — "
          f"auto-resuming {model_key} from epoch {initial_epoch}.")
    scratch_path = "/kaggle/working/_resume_scratch.keras"
    with gzip.open(latest_ckpt, 'rb') as f_in, open(scratch_path, 'wb') as f_out:
        shutil.copyfileobj(f_in, f_out)
    model = tf.keras.models.load_model(scratch_path, custom_objects=custom_objects)
    Path(scratch_path).unlink(missing_ok=True)
else:
    model = create_advanced_3d_unet(input_shape=(*PATCH_SIZE, 4), num_classes=NUM_CLASSES)
    model.compile(optimizer=Adam(learning_rate=CONFIG["learning_rate"]),
                   loss=loss_fn, metrics=train_metrics)
model.summary()

early_stop = EarlyStopping(monitor='val_loss', patience=20, mode='min', restore_best_weights=True)
best_checkpoint = ModelCheckpoint(
    BEST_MODEL_PATH, monitor='val_dice_whole_tumor', mode='max',
    save_best_only=True, verbose=1,
)
lr_scheduler = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=8, min_lr=1e-7, verbose=1)

class RotatingCheckpoint(tf.keras.callbacks.Callback):
    def __init__(self, path_pattern, keep_last=1):
        super().__init__()
        self.path_pattern = path_pattern
        self.keep_last = keep_last
        self.saved_paths = []

    def on_epoch_end(self, epoch, logs=None):
        gz_path = self.path_pattern.format(epoch=epoch + 1)
        raw_path = gz_path[:-3]
        self.model.save(raw_path)
        with open(raw_path, 'rb') as f_in, gzip.open(gz_path, 'wb') as f_out:
            shutil.copyfileobj(f_in, f_out)
        Path(raw_path).unlink(missing_ok=True)

        self.saved_paths.append(gz_path)
        while len(self.saved_paths) > self.keep_last:
            old_path = self.saved_paths.pop(0)
            try:
                Path(old_path).unlink(missing_ok=True)
            except Exception as e:
                print(f"Could not delete old checkpoint {old_path}: {e}")

per_epoch_full_checkpoint = RotatingCheckpoint(EPOCH_CKPT_PATTERN, keep_last=1)

# --- NEW: live Excel logging callbacks ------------------------------
epoch_log_logger = EpochMetricsExcelLogger("/kaggle/working/unet_training_epoch_log.xlsx")
periodic_val_eval_logger = PeriodicFullVolumeEvalLogger(
    val_df, PATCH_SIZE, STRIDE, NUM_CLASSES,
    excel_path="/kaggle/working/unet_val_results_LIVE.xlsx",
    eval_every=EVAL_EVERY,
)

callbacks = [
    early_stop, best_checkpoint, lr_scheduler, per_epoch_full_checkpoint,
    epoch_log_logger, periodic_val_eval_logger,
]

if initial_epoch >= EPOCHS:
    print(f"{model_key}: existing checkpoint is already at epoch "
          f"{initial_epoch} >= EPOCHS={EPOCHS}; skipping further training.")
    history = None
else:
    history = model.fit(
        train_gen, validation_data=val_gen, epochs=EPOCHS,
        initial_epoch=initial_epoch, callbacks=callbacks,
    )


Training architecture: unet
Found existing checkpoint '/kaggle/working/unet_full_epoch_040.keras.gz' — auto-resuming unet from epoch 40.


I0000 00:00:1788749554.588254      57 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1788749554.591310      57 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Model: "Advanced_3D_U-Net"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 112, 112,  │          0 │ -                 │
│ (InputLayer)        │ 112, 4)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv3d_19 (Conv3D)  │ (None, 112, 112,  │      3,488 │ input_layer_1[0]… │
│                     │ 112, 32)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 112, 112,  │        128 │ conv3d_19[0][0]   │
│ (BatchNormalizatio… │ 112, 32)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_18       │ (None, 112, 112,  │          0 │ batch_normalizat… │
│ (Activation)        │ 112, 32)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv3d_20 (Conv3D)  │ (None, 112, 112,  │     27,680 │ activation_18[0]… │
│                     │ 112, 32)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 112, 112,  │        128 │ conv3d_20[0][0]   │
│ (BatchNormalizatio… │ 112, 32)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_19       │ (None, 112, 112,  │          0 │ batch_normalizat… │
│ (Activation)        │ 112, 32)          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling3d_4     │ (None, 56, 56,    │          0 │ activation_19[0]… │
│ (MaxPooling3D)      │ 56, 32)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ spatial_dropout3d_4 │ (None, 56, 56,    │          0 │ max_pooling3d_4[… │
│ (SpatialDropout3D)  │ 56, 32)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv3d_21 (Conv3D)  │ (None, 56, 56,    │     55,360 │ spatial_dropout3… │
│                     │ 56, 64)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 56, 56,    │        256 │ conv3d_21[0][0]   │
│ (BatchNormalizatio… │ 56, 64)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_20       │ (None, 56, 56,    │          0 │ batch_normalizat… │
│ (Activation)        │ 56, 64)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv3d_22 (Conv3D)  │ (None, 56, 56,    │    110,656 │ activation_20[0]… │
│                     │ 56, 64)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 56, 56,    │        256 │ conv3d_22[0][0]   │
│ (BatchNormalizatio… │ 56, 64)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_21       │ (None, 56, 56,    │          0 │ batch_normalizat… │
│ (Activation)        │ 56, 64)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling3d_5     │ (None, 28, 28,    │          0 │ activation_21[0]… │
│ (MaxPooling3D)      │ 28, 64)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ spatial_dropout3d_5 │ (None, 28, 28,    │          0 │ max_pooling3d_5[

 Total params: 77,680,174 (296.33 MB)

 Trainable params: 25,891,428 (98.77 MB)

 Non-trainable params: 5,888 (23.00 KB)

 Optimizer params: 51,782,858 (197.54 MB)

unet: existing checkpoint is already at epoch 40 >= EPOCHS=40; skipping further training.


In [26]:
val_final_path = f"/kaggle/working/{model_key}_val_results_FINAL.xlsx"
test_final_path = f"/kaggle/working/{model_key}_test_results_FINAL.xlsx"

if os.path.exists(val_final_path) and os.path.exists(test_final_path):
    print(f"Final evaluation outputs already exist "
          f"('{val_final_path}', '{test_final_path}') — skipping re-evaluation.")
    print("Delete these files (or rename them) first if you actually want to re-run "
          "the final evaluation, e.g. after changing STRIDE/PATCH_SIZE or retraining.")
    test_summary = pd.read_excel(test_final_path, sheet_name='summary', index_col=0)
else:
    best_model = tf.keras.models.load_model(BEST_MODEL_PATH, custom_objects=custom_objects)

    print("Running final sliding-window full-volume evaluation on VAL set...")
    val_results = evaluate_full_dataset(best_model, val_df, PATCH_SIZE, STRIDE, NUM_CLASSES)
    val_summary = summarize_results(val_results)
    save_results_excel(val_final_path, val_results, val_summary)

    print("Running final sliding-window full-volume evaluation on TEST set...")
    test_results = evaluate_full_dataset(best_model, test_df, PATCH_SIZE, STRIDE, NUM_CLASSES)
    test_summary = summarize_results(test_results)
    save_results_excel(test_final_path, test_results, test_summary)

print(f"\n[{model_key}] Test set summary (mean / std / median across patients):")
print(test_summary)
print(f"\n[{model_key}] Test set — Region-wise summary table:")
print(format_region_metric_table(test_summary, use_std=True))

Running final sliding-window full-volume evaluation on VAL set...
Running final sliding-window full-volume evaluation on TEST set...

[unet] Test set summary (mean / std / median across patients):
                  mean       std    median
WT_Dice       0.897435  0.136544  0.940419
WT_IoU        0.833670  0.163435  0.887540
WT_Precision  0.922597  0.091480  0.946410
WT_Recall     0.899914  0.156897  0.954855
TC_Dice       0.871091  0.214735  0.948068
TC_IoU        0.814087  0.228348  0.901263
TC_Precision  0.944804  0.105257  0.977507
TC_Recall     0.851955  0.229429  0.933836
ET_Dice       0.860364  0.181692  0.915058
ET_IoU        0.784957  0.194041  0.843416
ET_Precision  0.886231  0.158571  0.934819
ET_Recall     0.870310  0.177392  0.930795

[unet] Test set — Region-wise summary table:
                                 Dice              IoU        Precision  \
Region                                                                    
Whole Tumor (WT)      0.8974 ± 0.1365  0.8337 ± 

In [27]:
import pandas as pd

def display_summary_table(summary_df, split_name):
    print(f"\n=======================================================")
    print(f"📊 FINAL PER-REGION SUMMARY FOR: {split_name.upper()} SET")
    print(f"=======================================================")
    
    # Check if summary is a dictionary or a complex DataFrame and format it cleanly
    if isinstance(summary_df, pd.DataFrame):
        # Displays a beautiful, readable ASCII table in Kaggle/Jupyter logs
        print(summary_df.to_string()) 
    else:
        # Fallback if your summary is a nested dictionary
        for region, metrics in summary_df.items():
            print(f"\n➔ Region: {region}")
            for metric_name, val in metrics.items():
                print(f"  {metric_name:<12}: {val}")

# Print validation summary
display_summary_table(val_summary, "validation")

# Print test summary
display_summary_table(test_summary, "test")



📊 FINAL PER-REGION SUMMARY FOR: VALIDATION SET
                  mean       std    median
WT_Dice       0.905460  0.104242  0.936161
WT_IoU        0.839528  0.133152  0.879985
WT_Precision  0.933780  0.075278  0.957679
WT_Recall     0.897518  0.134634  0.943662
TC_Dice       0.853682  0.255984  0.955140
TC_IoU        0.802233  0.262091  0.914132
TC_Precision  0.947953  0.133814  0.977958
TC_Recall     0.832309  0.267684  0.942421
ET_Dice       0.840524  0.220290  0.919890
ET_IoU        0.767081  0.227576  0.851664
ET_Precision  0.900299  0.155579  0.949096
ET_Recall     0.845767  0.217922  0.922898

📊 FINAL PER-REGION SUMMARY FOR: TEST SET
                  mean       std    median
WT_Dice       0.897435  0.136544  0.940419
WT_IoU        0.833670  0.163435  0.887540
WT_Precision  0.922597  0.091480  0.946410
WT_Recall     0.899914  0.156897  0.954855
TC_Dice       0.871091  0.214735  0.948068
TC_IoU        0.814087  0.228348  0.901263
TC_Precision  0.944804  0.105257  0.977507
TC_Reca